# Sparkle Movie — Recommandation de films avec PySpark

## Objectif
Explorer les données MovieLens et construire un système de recommandation
de films à partir des notes attribuées par les utilisateurs.

## Étapes
1. Préparer l’environnement.
2. Charger et comprendre les données.
3. Vérifier et nettoyer les données.
4. Explorer les notes et les films.
5. Entraîner un modèle ALS avec Spark ML.
6. Évaluer le modèle et produire des recommandations.

In [1]:
%pip install -q pyspark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SparkleMovie")
    .master("local[2]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Version de Spark :", spark.version)

Version de Spark : 4.0.5


## Configuration de Spark

Nous créons une session Spark nommée « SparkleMovie ».

Le traitement s’exécute sur la machine mise à disposition par Google Colab.
Nous utilisons le mode local avec deux threads de travail.

La variable `spark` permettra ensuite de charger et de traiter les données.

In [7]:
movies = (
    spark.read
    .option("header", True)
    .schema("movieId INT, title STRING, genres STRING")
    .csv("/content/movies.csv")
)

movies.show(10, truncate=False)

+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
|6      |Heat (1995)                       |Action|Crime|Thriller                      |
|7      |Sabrina (1995)                    |Comedy|Romance                             |
|8      |Tom and Huck (1995)               |Adventure|Children                         |
|9      |Sudden Death

In [8]:
ratings = (
    spark.read
    .option("header", True)
    .schema(
        "userId INT, movieId INT, rating DOUBLE, timestamp BIGINT"
    )
    .csv("/content/ratings.csv")
)

ratings.show(10)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     17|   4.0|944249077|
|     1|     25|   1.0|944250228|
|     1|     29|   2.0|943230976|
|     1|     30|   5.0|944249077|
|     1|     32|   5.0|943228858|
|     1|     34|   2.0|943228491|
|     1|     36|   1.0|944249008|
|     1|     80|   5.0|944248943|
|     1|    110|   3.0|943231119|
|     1|    111|   5.0|944249008|
+------+-------+------+---------+
only showing top 10 rows


In [9]:
print("STRUCTURE DES FILMS")
movies.printSchema()

print("STRUCTURE DES NOTES")
ratings.printSchema()

STRUCTURE DES FILMS
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

STRUCTURE DES NOTES
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)



## Vérification de la qualité des données

Avant de réaliser les analyses, nous recherchons les valeurs manquantes
et les lignes en double dans les tableaux des films et des notes.

Nous déciderons du nettoyage à effectuer à partir des résultats.

In [10]:
nombre_films = movies.count()
nombre_notes = ratings.count()

print(f"Nombre de lignes dans movies : {nombre_films:,}")
print(f"Nombre de lignes dans ratings : {nombre_notes:,}")

Nombre de lignes dans movies : 87,585
Nombre de lignes dans ratings : 2,203,609


In [11]:
from pyspark.sql import functions as F

print("VALEURS MANQUANTES DANS MOVIES")

movies.select([
    F.count(
        F.when(F.col(colonne).isNull(), 1)
    ).alias(colonne)
    for colonne in movies.columns
]).show()

print("VALEURS MANQUANTES DANS RATINGS")

ratings.select([
    F.count(
        F.when(F.col(colonne).isNull(), 1)
    ).alias(colonne)
    for colonne in ratings.columns
]).show()

VALEURS MANQUANTES DANS MOVIES
+-------+-----+------+
|movieId|title|genres|
+-------+-----+------+
|      0|    0|     0|
+-------+-----+------+

VALEURS MANQUANTES DANS RATINGS
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     0|      0|     0|        0|
+------+-------+------+---------+



In [12]:
doublons_films = nombre_films - movies.dropDuplicates().count()
doublons_notes = nombre_notes - ratings.dropDuplicates().count()

print("Lignes en double dans movies :", doublons_films)
print("Lignes en double dans ratings :", doublons_notes)

Lignes en double dans movies : 0
Lignes en double dans ratings : 0


### Résultat des premiers contrôles de qualité

Le tableau des films contient 87 585 lignes et celui des évaluations
contient 2 203 609 lignes.

Aucune valeur nulle et aucun doublon exact n’ont été détectés
dans les deux DataFrames.

Aucune suppression n’est donc effectuée à ce stade.
Nous poursuivons les vérifications sur la validité des valeurs.

In [13]:
from pyspark.sql import functions as F

repartition_notes = (
    ratings
    .groupBy("rating")
    .agg(F.count("*").alias("nombre_notes"))
    .orderBy("rating")
)

repartition_notes.show(20)

+------+------------+
|rating|nombre_notes|
+------+------------+
|   0.5|       39164|
|   1.0|       67006|
|   1.5|       35312|
|   2.0|      136557|
|   2.5|      114968|
|   3.0|      412940|
|   3.5|      296718|
|   4.0|      575899|
|   4.5|      208527|
|   5.0|      316518|
+------+------------+



In [14]:
films_champs_vides = movies.filter(
    (F.trim(F.col("title")) == "") |
    (F.trim(F.col("genres")) == "")
)

print(
    "Films avec un titre ou un champ genres vide :",
    films_champs_vides.count()
)

films_champs_vides.show(5, truncate=False)

Films avec un titre ou un champ genres vide : 0
+-------+-----+------+
|movieId|title|genres|
+-------+-----+------+
+-------+-----+------+



### Conclusion des contrôles effectués

Les contrôles réalisés n’ont détecté aucune valeur nulle,
aucun doublon exact et aucun titre ou champ genres vide.

Les notes observées vont de 0,5 à 5, par pas de 0,5.

Aucune suppression de ligne n’est effectuée à ce stade.
Ces contrôles ne garantissent toutefois pas, à eux seuls,
l’absence de toute autre anomalie.

## Analyse des films les mieux notés

Nous calculons la note moyenne et le nombre d’évaluations de chaque film.

Nous examinerons d’abord le classement sans seuil, puis un classement
limité aux films ayant reçu au moins 100 évaluations.
Ce seuil est un choix d’analyse, et non une exigence du sujet.